In [83]:
import pandas as pd
import numpy as np
import torch
import torch.nn as nn
from sklearn.preprocessing import StandardScaler, MinMaxScaler
import os
import time






In [84]:

DATA_DIR = "dataset"
OUTPUT_DIR = "research\models\impute_result"
MODEL_PATH = "deep_learning_models/grud_model.pt" 

device = torch.device('mps' if torch.backends.mps.is_available() else 'cpu')

In [86]:
train_data = pd.read_pickle(os.path.join(DATA_DIR, 'train_data_final.pkl'))
val_data = pd.read_pickle(os.path.join(DATA_DIR, 'val_data.pkl'))
test_data = pd.read_pickle(os.path.join(DATA_DIR, 'test_data.pkl'))

print(f"  Train: {train_data.shape}")
print(f"  Val:   {val_data.shape}")
print(f"  Test:  {test_data.shape}")

  Train: (143459, 75)
  Val:   (30741, 75)
  Test:  (30742, 75)


In [87]:
missing_cols = [c for c in train_data.columns if c.endswith('_missing')]

In [ ]:
# Drop columns 
drop_cols = ['PCell_Cell_ID', 'PCell_Uplink_frequency']
train_data = train_data.drop(columns=[c for c in drop_cols if c in train_data.columns])
val_data = val_data.drop(columns=[c for c in drop_cols if c in val_data.columns])
test_data = test_data.drop(columns=[c for c in drop_cols if c in test_data.columns])

In [90]:
print(f"  Train: {train_data.shape}")
print(f"  Val:   {val_data.shape}")
print(f"  Test:  {test_data.shape}")

  Train: (143459, 75)
  Val:   (30741, 75)
  Test:  (30742, 75)


In [91]:
train_data.columns

Index(['timestamp', 'ping_ms', 'datarate', 'jitter', 'Latitude', 'Longitude',
       'Altitude', 'speed_kmh', 'COG', 'precipIntensity', 'precipProbability',
       'temperature', 'humidity', 'windSpeed', 'Traffic Jam Factor',
       'Traffic Distance', 'Pos in Ref Round', 'measurement', 'area',
       'PCell_RSRP_max', 'PCell_RSRQ_max', 'PCell_RSSI_max', 'PCell_SNR_1',
       'PCell_SNR_2', 'PCell_Downlink_Num_RBs', 'PCell_Downlink_TB_Size',
       'PCell_Downlink_Average_MCS', 'PCell_Uplink_Num_RBs',
       'PCell_Uplink_TB_Size', 'PCell_Uplink_Tx_Power_(dBm)',
       'PCell_Downlink_frequency', 'PCell_Downlink_bandwidth_MHz',
       'PCell_Uplink_bandwidth_MHz', 'PCell_Band_Indicator', 'PCell_freq_MHz',
       'scenario', 'target_datarate', 'operator', 'PCell_DL_RBs_MCS_Low',
       'PCell_DL_RBs_MCS_Mid', 'PCell_DL_RBs_MCS_High', 'ping_ms_missing',
       'datarate_missing', 'jitter_missing', 'Pos in Ref Round_missing',
       'PCell_Cell_Identity_missing', 'PCell_Band_Indicator_mis

In [ ]:

# Training Features
continuous_impute = [
    'ping_ms', 'datarate', 'jitter', 'Latitude', 'Longitude',
    'Altitude', 'speed_kmh', 'COG', 'precipIntensity',
    'precipProbability', 'temperature', 'humidity', 'windSpeed',
    'Traffic Jam Factor', 'Traffic Distance', 'Pos in Ref Round',
    'measurement', 'PCell_RSRP_max', 'PCell_RSRQ_max',
    'PCell_RSSI_max', 'PCell_SNR_1', 'PCell_SNR_2',
    'PCell_Downlink_Num_RBs', 'PCell_Downlink_TB_Size',
    'PCell_Downlink_Average_MCS', 'PCell_Uplink_Num_RBs',
    'PCell_Uplink_TB_Size', 'PCell_Uplink_Tx_Power_(dBm)',
    'PCell_Downlink_bandwidth_MHz', 'PCell_Uplink_bandwidth_MHz',
    'PCell_Band_Indicator', 'target_datarate',
    'PCell_DL_RBs_MCS_Low', 'PCell_DL_RBs_MCS_Mid',
    'PCell_DL_RBs_MCS_High',
]

log_features = [
    'datarate', 'target_datarate',
    'PCell_Downlink_TB_Size', 'PCell_Uplink_TB_Size',
    'PCell_DL_RBs_MCS_Low', 'PCell_DL_RBs_MCS_Mid',
    'PCell_DL_RBs_MCS_High',
    'PCell_Downlink_Num_RBs', 'PCell_Uplink_Num_RBs',
    'ping_ms', 'Pos in Ref Round', 'Traffic Distance',
]

freq_cols = ['PCell_freq_MHz','PCell_Downlink_frequency']

static_features = [
    'operator',
    'device_pc1', 'device_pc2', 'device_pc3', 'device_pc4',
    'direction_uplink', 'measured_qos_delay',
    'hour', 'day_of_week',
]

In [93]:
SEQ_LEN = 20
BATCH_SIZE = 256

In [94]:
def encode_cell_freq(df, dummy_columns=None):
    df = df.copy()
    for col in freq_cols:
        df[col] = df[col].fillna(-1)
    dummies = pd.get_dummies(
        df[freq_cols],
        columns=freq_cols,
        prefix=['band','PCell_DL_frequency'],
        dtype=int
    )
    if dummy_columns is not None:
        dummies = dummies.reindex(columns=dummy_columns, fill_value=0)
    return df, dummies, dummies.columns.tolist()


def clip_outliers_per_feature(data, features, reference_data=None,
                               lower_pct=0.5, upper_pct=99.5):
    data_clipped = data.copy()
    clipping_bounds = {}
    for feat in features:
        if feat not in data.columns:
            continue
        ref_col = (reference_data[feat] if reference_data is not None
                   else data[feat]).dropna()
        if len(ref_col) == 0:
            continue
        lower_bound = np.percentile(ref_col, lower_pct)
        upper_bound = np.percentile(ref_col, upper_pct)
        clipping_bounds[feat] = (lower_bound, upper_bound)
        mask = data_clipped[feat].notna()
        data_clipped.loc[mask, feat] = data_clipped.loc[mask, feat].clip(
            lower=lower_bound, upper=upper_bound
        )
    return data_clipped, clipping_bounds


def compute_time_deltas(timestamps):
    ts = pd.to_datetime(timestamps)
    deltas = ts.diff().dt.total_seconds().fillna(0).values
    deltas = np.clip(deltas, 0, 60)
    deltas = deltas / 60.0
    return deltas


def create_sequences(scaled_data, static_data, deltas, seq_len):
    n_samples = len(scaled_data) - seq_len + 1
    n_features = scaled_data.shape[1]
    n_static = static_data.shape[1]

    mask = (~np.isnan(scaled_data)).astype(np.float32)
    data_filled = np.nan_to_num(scaled_data, nan=0.0)

    sequences = np.zeros((n_samples, seq_len, n_features), dtype=np.float32)
    masks = np.zeros((n_samples, seq_len, n_features), dtype=np.float32)
    time_gaps = np.zeros((n_samples, seq_len), dtype=np.float32)
    statics = np.zeros((n_samples, n_static), dtype=np.float32)

    for i in range(n_samples):
        sequences[i] = data_filled[i:i+seq_len]
        masks[i] = mask[i:i+seq_len]
        time_gaps[i] = deltas[i:i+seq_len]
        statics[i] = static_data[i+seq_len-1]

    return sequences, masks, time_gaps, statics

In [95]:
# Clipping bounds from train
_, clipping_bounds = clip_outliers_per_feature(train_data, continuous_impute)
train_clipped, _ = clip_outliers_per_feature(train_data, continuous_impute)

# Categorical encoding — get dummy column names from train
_, train_dummies, dummy_cols = encode_cell_freq(train_data)

# Fit continuous scaler on train
cont_scaler = StandardScaler()
cont_scaler.fit(train_clipped[continuous_impute].values)

# Fit static scaler on train
static_scaler = MinMaxScaler()
static_scaler.fit(train_data[static_features].fillna(0).values)

n_continuous = len(continuous_impute)
n_cat_encoded = len(dummy_cols)
n_input = n_continuous + n_cat_encoded
n_static = len(static_features)

print(f"  Input dim: {n_input} ({n_continuous} cont + {n_cat_encoded} cat)")
print(f"  Static dim: {n_static}")
print(f"  Output dim: {n_continuous}")

  Input dim: 51 (35 cont + 16 cat)
  Static dim: 9
  Output dim: 35


In [49]:
class GRUD(nn.Module):
    def __init__(self, input_dim, static_dim, hidden_dim=128, output_dim=None):
        super(GRUD, self).__init__()
        self.input_dim = input_dim
        self.hidden_dim = hidden_dim
        self.output_dim = output_dim or input_dim

        self.W_gamma_x = nn.Linear(input_dim, input_dim)
        self.W_gamma_h = nn.Linear(input_dim, hidden_dim)
        self.x_mean = nn.Parameter(torch.zeros(input_dim))

        gru_input_dim = input_dim * 2 + static_dim
        self.W_z = nn.Linear(gru_input_dim + hidden_dim, hidden_dim)
        self.W_r = nn.Linear(gru_input_dim + hidden_dim, hidden_dim)
        self.W_h = nn.Linear(gru_input_dim + hidden_dim, hidden_dim)

        self.output_layer = nn.Sequential(
            nn.Linear(hidden_dim, hidden_dim),
            nn.ReLU(),
            nn.Dropout(0.3),
            nn.Linear(hidden_dim, self.output_dim)
        )

    def forward(self, x, mask, time_gaps, static):
        batch_size, seq_len, _ = x.shape
        h = torch.zeros(batch_size, self.hidden_dim, device=x.device)

        for t in range(seq_len):
            x_t = x[:, t, :]
            m_t = mask[:, t, :]
            dt = time_gaps[:, t:t+1]

            gamma_x = torch.exp(-torch.relu(
                self.W_gamma_x(dt.expand(-1, self.input_dim))
            ))
            gamma_h = torch.exp(-torch.relu(
                self.W_gamma_h(dt.expand(-1, self.input_dim))
            ))

            x_decayed = (m_t * x_t +
                         (1 - m_t) * (gamma_x * x_t +
                                      (1 - gamma_x) * self.x_mean))
            h = gamma_h * h

            combined = torch.cat([x_decayed, m_t, static], dim=1)
            combined_h = torch.cat([combined, h], dim=1)

            z = torch.sigmoid(self.W_z(combined_h))
            r = torch.sigmoid(self.W_r(combined_h))
            h_tilde_input = torch.cat([combined, r * h], dim=1)
            h_tilde = torch.tanh(self.W_h(h_tilde_input))
            h = (1 - z) * h + z * h_tilde

        output = self.output_layer(h)
        return output

In [50]:
model = GRUD(n_input, n_static, hidden_dim=128, output_dim=n_continuous).to(device)

# Load model weights
# Option A: If you saved model.state_dict()
if os.path.exists(MODEL_PATH):
    checkpoint = torch.load(MODEL_PATH, map_location=device, weights_only=False)
    if isinstance(checkpoint, dict) and 'model_state_dict' in checkpoint:
        model.load_state_dict(checkpoint['model_state_dict'])
    elif isinstance(checkpoint, dict) and not any(
        k.startswith('W_') or k.startswith('output') for k in checkpoint.keys()
    ):
        # checkpoint is a full checkpoint dict with other keys
        model.load_state_dict(checkpoint)
    else:
        model.load_state_dict(checkpoint)
    print(f"  Loaded from: {MODEL_PATH}")
else:
    print(f"  ERROR: Model file not found at {MODEL_PATH}")
    print(f"  Please update MODEL_PATH in the script")
    print(f"\n  If you have best_model_state in memory, save it first:")
    print(f"    torch.save(best_model_state, '{MODEL_PATH}')")
    



  Loaded from: deep_learning_models/grud_model.pt


In [ ]:
def impute_split(model, df_original, cont_scaler, clipping_bounds,
                 dummy_cols, static_scaler, train_data_ref, device):
    """
    Run GRU-D on a data split and fill in missing values.
    Only replaces values that are ACTUALLY missing
    """
    
    missing_mask = {}
    for col in continuous_impute:
        if col in df_original.columns:
            missing_mask[col] = df_original[col].isna()

    
    # Clip
    df_clipped, _ = clip_outliers_per_feature(
        df_original, continuous_impute, reference_data=train_data_ref
    )

    # Encode categoricals
    _, dummies, _ = encode_cell_freq(df_clipped, dummy_columns=dummy_cols)

    # Scale continuous
    cont_scaled = cont_scaler.transform(df_clipped[continuous_impute].values)

    # Categorical encoded
    cat_enc = dummies.values

    # Full input
    full_scaled = np.hstack([cont_scaled, cat_enc])

    # Static
    static_df = df_original[static_features].fillna(0)
    static_scaled = static_scaler.transform(static_df.values)

    # Time deltas
    deltas = compute_time_deltas(df_original['timestamp'])

    # Create sequences
    sequences, masks, time_gaps, statics = create_sequences(
        full_scaled, static_scaled, deltas, SEQ_LEN
    )

    # Run model
    model.eval()
    all_predictions = []

    dataset = torch.utils.data.TensorDataset(
        torch.FloatTensor(sequences),
        torch.FloatTensor(masks),
        torch.FloatTensor(time_gaps),
        torch.FloatTensor(statics),
    )
    loader = torch.utils.data.DataLoader(
        dataset, batch_size=BATCH_SIZE, shuffle=False
    )

    with torch.no_grad():
        for seq, mask, gaps, static in loader:
            seq = seq.to(device)
            mask = mask.to(device)
            gaps = gaps.to(device)
            static = static.to(device)

            outputs = model(seq, mask, gaps, static)
            all_predictions.append(outputs.cpu().numpy())

    predictions_scaled = np.vstack(all_predictions)

    # Inverse transform: scaled → original scale
    predictions_original = cont_scaler.inverse_transform(predictions_scaled)

    # Fill missing values only
    df_imputed = df_original.copy()
    pred_start_idx = SEQ_LEN - 1
    pred_indices = df_original.index[pred_start_idx:pred_start_idx + len(predictions_original)]

    n_filled = 0
    for i, col in enumerate(continuous_impute):
        if col not in missing_mask:
            continue

        for j, idx in enumerate(pred_indices):
            if missing_mask[col].loc[idx]:  # Only fill if originally missing
                df_imputed.loc[idx, col] = predictions_original[j, i]
                n_filled += 1

    # Handle first SEQ_LEN-1 rows (no sequence history)
    # Use forward fill from the nearest predicted value
    for col in continuous_impute:
        if col not in missing_mask:
            continue

        first_rows_missing = missing_mask[col].iloc[:pred_start_idx]
        if first_rows_missing.any():
            # Forward fill from the first predicted value, or use column median
            first_valid = df_imputed[col].iloc[pred_start_idx:].first_valid_index()
            if first_valid is not None:
                fill_val = df_imputed.loc[first_valid, col]
            else:
                fill_val = df_original[col].median()

            for idx in df_original.index[:pred_start_idx]:
                if missing_mask[col].loc[idx]:
                    df_imputed.loc[idx, col] = fill_val

    return df_imputed, n_filled

In [52]:
# Set x_mean from train data
with torch.no_grad():
    train_enc_temp, train_dum_temp, _ = encode_cell_freq(train_data, dummy_cols)
    train_cont_temp = cont_scaler.transform(
        train_clipped[continuous_impute].values
    )
    train_cat_temp = train_dum_temp.values
    train_full = np.hstack([train_cont_temp, train_cat_temp])
    train_means = np.nanmean(train_full, axis=0)
    model.x_mean.copy_(torch.FloatTensor(train_means))

model.eval()
print(f"  Model loaded successfully")

  Model loaded successfully


In [53]:
t0 = time.time()
train_imputed, n_train = impute_split(
    model, train_data, cont_scaler, clipping_bounds,
    dummy_cols, static_scaler, train_data, device
)
print(f"  Filled {n_train:,} values in {time.time()-t0:.1f}s")

  Filled 231,042 values in 164.6s


In [54]:
t0 = time.time()
val_imputed, n_val = impute_split(
    model, val_data, cont_scaler, clipping_bounds,
    dummy_cols, static_scaler, train_data, device
)
print(f"  Filled {n_val:,} values in {time.time()-t0:.1f}s")

  Filled 11,074 values in 10.6s


In [55]:
t0 = time.time()
test_imputed, n_test = impute_split(
    model, test_data, cont_scaler, clipping_bounds,
    dummy_cols, static_scaler, train_data, device
)
print(f"  Filled {n_test:,} values in {time.time()-t0:.1f}s")

  Filled 20,138 values in 12.5s


In [ ]:


df_complete = pd.concat([train_imputed, val_imputed, test_imputed])
df_complete = df_complete.sort_values('timestamp').reset_index(drop=True)

# Verify: check remaining NaN in continuous features
print(f"\nComplete dataset: {df_complete.shape}")
print(f"\nRemaining NaN in continuous features:")
remaining_nan = 0
for col in continuous_impute:
    n_nan = df_complete[col].isna().sum()
    if n_nan > 0:
        print(f"  {col}: {n_nan:,} NaN remaining")
        remaining_nan += n_nan



COMBINING INTO COMPLETE DATASET

Complete dataset: (204942, 75)

Remaining NaN in continuous features:


In [57]:
remaining_nan

0

In [58]:
if remaining_nan == 0:
    print(f"  ZERO NaN — dataset is fully imputed!")
else:
    print(f"\n  Total remaining NaN: {remaining_nan:,}")
    print(f"  These are likely in the first {SEQ_LEN-1} rows per split")
    print(f"  Filling remaining with forward/backward fill...")

    for col in continuous_impute:
        if df_complete[col].isna().any():
            df_complete[col] = df_complete[col].ffill().bfill()

    final_nan = df_complete[continuous_impute].isna().sum().sum()
    print(f"  After ffill/bfill: {final_nan} NaN remaining")

  ZERO NaN — dataset is fully imputed!


In [62]:
df_original = pd.read_parquet('results/df_engineered.parquet')

In [63]:
remove_from_imputation = [
    'PCell_Cell_Identity',    
    'PCell_TAC',                 
    'PCell_E-ARFCN','PCell_Uplink_frequency','PCell_Cell_ID'
]

In [27]:
output_path = os.path.join(OUTPUT_DIR, "df_complete_imputed.pkl")
df_complete.to_pickle(output_path)
print(f"\nSaved: {output_path}")

# Also save the splits separately
train_imputed.to_pickle(os.path.join(OUTPUT_DIR, "train_imputed_grud.pkl"))
val_imputed.to_pickle(os.path.join(OUTPUT_DIR, "val_imputed_grud.pkl"))
test_imputed.to_pickle(os.path.join(OUTPUT_DIR, "test_imputed_grud.pkl"))
print(f"Saved individual splits too")


Saved: research\models\impute_result\df_complete_imputed.pkl
Saved individual splits too


In [ ]:
df = pd.read_pickle('research\models\impute_result\df_complete_imputed.pkl')

print(f"Shape: {df.shape}")
print(f"\nColumns:\n{list(df.columns)}")
print(f"\nAny NaN: {df.isna().sum().sum()}")
print(f"\nTimestamp range: {df['timestamp'].min()} to {df['timestamp'].max()}")
print(f"\nDevice distribution:")
for d in ['device_pc1', 'device_pc2', 'device_pc3', 'device_pc4']:
    if d in df.columns:
        print(f"  {d}: {df[d].sum():.0f} rows")

Shape: (204942, 75)

Columns:
['timestamp', 'ping_ms', 'datarate', 'jitter', 'Latitude', 'Longitude', 'Altitude', 'speed_kmh', 'COG', 'precipIntensity', 'precipProbability', 'temperature', 'humidity', 'windSpeed', 'Traffic Jam Factor', 'Traffic Distance', 'Pos in Ref Round', 'measurement', 'area', 'PCell_RSRP_max', 'PCell_RSRQ_max', 'PCell_RSSI_max', 'PCell_SNR_1', 'PCell_SNR_2', 'PCell_Downlink_Num_RBs', 'PCell_Downlink_TB_Size', 'PCell_Downlink_Average_MCS', 'PCell_Uplink_Num_RBs', 'PCell_Uplink_TB_Size', 'PCell_Uplink_Tx_Power_(dBm)', 'PCell_Downlink_frequency', 'PCell_Downlink_bandwidth_MHz', 'PCell_Uplink_bandwidth_MHz', 'PCell_Band_Indicator', 'PCell_freq_MHz', 'scenario', 'target_datarate', 'operator', 'PCell_DL_RBs_MCS_Low', 'PCell_DL_RBs_MCS_Mid', 'PCell_DL_RBs_MCS_High', 'ping_ms_missing', 'datarate_missing', 'jitter_missing', 'Pos in Ref Round_missing', 'PCell_Cell_Identity_missing', 'PCell_Band_Indicator_missing', 'PCell_Uplink_frequency_missing', 'PCell_Downlink_bandwidth_

In [ ]:
# Run this to see which columns still have NaN
nan_counts = df.isna().sum()
nan_cols = nan_counts[nan_counts > 0]
print(f"Columns with NaN ({len(nan_cols)}):")
for col, count in nan_cols.sort_values(ascending=False).items():
    print(f"  {col:<45} {count:>8,} ({count/len(df)*100:.2f}%)")

Columns with NaN (2):
  PCell_Downlink_frequency                        12,607 (6.15%)
  PCell_freq_MHz                                  10,031 (4.89%)


In [ ]:
for col in df.select_dtypes(include=['float64', 'float32']).columns:
    unique_count = df[col].nunique()
    if unique_count > 20:
        sample = df[col].dropna().head(1000)
        is_integer = (sample == sample.round()).all()
        if not is_integer:
            pct_round = (sample == sample.round()).mean()
            if pct_round > 0.5 and pct_round < 1.0:
                print(f"{col}:")
                print(f"  Unique: {unique_count}")
                print(f"  % integer-like: {pct_round*100:.1f}%")
                print(f"  Sample non-integer values: {sorted(sample[sample != sample.round()].unique()[:5])}")
                print()

PCell_Downlink_Average_MCS:
  Unique: 10507
  % integer-like: 98.6%
  Sample non-integer values: [np.float64(9.4815673828125), np.float64(9.654659271240234), np.float64(9.881030082702637), np.float64(10.408317565917969), np.float64(10.444368362426758)]

PCell_Downlink_bandwidth_MHz:
  Unique: 12385
  % integer-like: 97.1%
  Sample non-integer values: [np.float64(16.497224807739258), np.float64(16.738506317138672), np.float64(16.751323699951172), np.float64(16.75681495666504), np.float64(16.858938217163086)]

PCell_Uplink_bandwidth_MHz:
  Unique: 12372
  % integer-like: 97.1%
  Sample non-integer values: [np.float64(16.497224807739258), np.float64(16.738506317138672), np.float64(16.751323699951172), np.float64(16.75681495666504), np.float64(16.858938217163086)]

PCell_Band_Indicator:
  Unique: 12590
  % integer-like: 97.1%
  Sample non-integer values: [np.float64(5.124086380004883), np.float64(5.176089763641357), np.float64(5.192701816558838), np.float64(5.2882256507873535), np.float64(

In [ ]:


# Find valid values from the integer-like rows
for col in ['PCell_Downlink_Average_MCS', 'PCell_Downlink_bandwidth_MHz',
            'PCell_Uplink_bandwidth_MHz', 'PCell_Band_Indicator']:
    
    # Get valid values
    all_vals = df[col].dropna()
    integer_vals = all_vals[all_vals == all_vals.round()]
    valid_set = sorted(integer_vals.unique())
    
    print(f"\n{col}:")
    print(f"  Valid values: {valid_set}")
    
    # Snap non-integer values to nearest valid value
    non_integer_mask = df[col] != df[col].round()
    n_fix = non_integer_mask.sum()
    
    if n_fix > 0:
        def snap_to_nearest(val):
            if val == round(val):
                return val
            return min(valid_set, key=lambda x: abs(x - val))
        
        df.loc[non_integer_mask, col] = df.loc[non_integer_mask, col].apply(snap_to_nearest)
        print(f"  Fixed {n_fix:,} values")
    
    # Verify
    remaining = (df[col] != df[col].round()).sum()
    print(f"  Remaining non-integer: {remaining}")


PCell_Downlink_Average_MCS:
  Valid values: [np.float64(0.0), np.float64(1.0), np.float64(2.0), np.float64(3.0), np.float64(4.0), np.float64(5.0), np.float64(6.0), np.float64(7.0), np.float64(8.0), np.float64(9.0), np.float64(10.0), np.float64(11.0), np.float64(12.0), np.float64(13.0), np.float64(14.0), np.float64(15.0), np.float64(16.0), np.float64(17.0), np.float64(18.0), np.float64(19.0), np.float64(20.0), np.float64(21.0), np.float64(22.0), np.float64(23.0), np.float64(24.0), np.float64(25.0), np.float64(26.0), np.float64(27.0), np.float64(28.0), np.float64(29.0)]
  Fixed 10,506 values
  Remaining non-integer: 0

PCell_Downlink_bandwidth_MHz:
  Valid values: [np.float64(5.0), np.float64(10.0), np.float64(15.0), np.float64(20.0)]
  Fixed 12,607 values
  Remaining non-integer: 0

PCell_Uplink_bandwidth_MHz:
  Valid values: [np.float64(5.0), np.float64(10.0), np.float64(15.0), np.float64(20.0)]
  Fixed 12,607 values
  Remaining non-integer: 0

PCell_Band_Indicator:
  Valid values: [n

In [ ]:
# Forward fill categorical columns per device
categorical_to_fill = ['PCell_Downlink_frequency', 'PCell_freq_MHz', 'operator']

device_series = pd.Series('unknown', index=df.index)
for d in ['device_pc1', 'device_pc2', 'device_pc3', 'device_pc4']:
    device_series[df[d] == 1] = d

for col in categorical_to_fill:
    n_before = df[col].isna().sum()
    df[col] = df.groupby(device_series)[col].transform(
        lambda x: x.ffill().bfill()
    )
    n_after = df[col].isna().sum()
    print(f"{col}: {n_before:,} → {n_after:,} NaN")



PCell_Downlink_frequency: 12,607 → 0 NaN
PCell_freq_MHz: 10,031 → 0 NaN
operator: 0 → 0 NaN


In [ ]:
# drop _missing indicator columns
missing_cols = [c for c in df.columns if c.endswith('_missing')]
df = df.drop(columns=missing_cols)

# Final check
print(f"\nShape: {df.shape}")
print(f"Total NaN: {df.isna().sum().sum()}")
print(f"\nColumns ({len(df.columns)}):")
print(list(df.columns))

# Save
df.to_pickle('research\models\impute_result/df_complete_clean.pkl')
print(f"\nSaved: df_complete_clean.pkl")


Shape: (204942, 50)
Total NaN: 0

Columns (50):
['timestamp', 'ping_ms', 'datarate', 'jitter', 'Latitude', 'Longitude', 'Altitude', 'speed_kmh', 'COG', 'precipIntensity', 'precipProbability', 'temperature', 'humidity', 'windSpeed', 'Traffic Jam Factor', 'Traffic Distance', 'Pos in Ref Round', 'measurement', 'area', 'PCell_RSRP_max', 'PCell_RSRQ_max', 'PCell_RSSI_max', 'PCell_SNR_1', 'PCell_SNR_2', 'PCell_Downlink_Num_RBs', 'PCell_Downlink_TB_Size', 'PCell_Downlink_Average_MCS', 'PCell_Uplink_Num_RBs', 'PCell_Uplink_TB_Size', 'PCell_Uplink_Tx_Power_(dBm)', 'PCell_Downlink_frequency', 'PCell_Downlink_bandwidth_MHz', 'PCell_Uplink_bandwidth_MHz', 'PCell_Band_Indicator', 'PCell_freq_MHz', 'scenario', 'target_datarate', 'operator', 'PCell_DL_RBs_MCS_Low', 'PCell_DL_RBs_MCS_Mid', 'PCell_DL_RBs_MCS_High', 'device_pc1', 'device_pc2', 'device_pc3', 'device_pc4', 'direction_uplink', 'measured_qos_delay', 'hour', 'day_of_week', 'date']

Saved: df_complete_clean.pkl


In [ ]:

featurewise_imputation_pct = pd.DataFrame({
    "feature": continuous_impute,
    "missing_in_original": [df_original[col].isna().sum() for col in continuous_impute],
    "missing_pct_in_original": [df_original[col].isna().mean() * 100 for col in continuous_impute],
    "missing_values": [df_complete[col].isna().sum() for col in continuous_impute],
    "missing_values_pct": [df_complete[col].isna().mean() * 100 for col in continuous_impute],
})

featurewise_imputation_pct["imputed_count"] = (
    featurewise_imputation_pct["missing_in_original"]
    - featurewise_imputation_pct["missing_values"]
)

featurewise_imputation_pct["imputed_pct_of_dataset"] = (
    featurewise_imputation_pct["imputed_count"] / len(df_original) * 100
)

featurewise_imputation_pct = featurewise_imputation_pct.sort_values(
    "missing_pct_in_original", ascending=False
).reset_index(drop=True)

print("Feature-wise missing/imputation percentage:")
display(featurewise_imputation_pct)

featurewise_imputation_pct.to_csv(
    os.path.join(OUTPUT_DIR, "featurewise_imputation_percentage.csv"),
    index=False
)


Feature-wise missing/imputation percentage:


,feature,missing_in_original,missing_pct_in_original,missing_values,missing_values_pct,imputed_count,imputed_pct_of_dataset
0,ping_ms,57707,28.157723,0,0.0,57707,28.157723
1,datarate,15679,7.650457,0,0.0,15679,7.650457
2,jitter,15679,7.650457,0,0.0,15679,7.650457
3,Pos in Ref Round,14159,6.908784,0,0.0,14159,6.908784
4,PCell_Band_Indicator,12607,6.151497,0,0.0,12607,6.151497
5,PCell_Uplink_bandwidth_MHz,12607,6.151497,0,0.0,12607,6.151497
6,PCell_Downlink_bandwidth_MHz,12607,6.151497,0,0.0,12607,6.151497
7,PCell_Uplink_Num_RBs,12210,5.957783,0,0.0,12210,5.957783
8,PCell_Uplink_TB_Size,12210,5.957783,0,0.0,12210,5.957783
9,PCell_Uplink_Tx_Power_(dBm),12210,5.957783,0,0.0,12210,5.957783
